In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

In [0]:
source = spark.table("default.groww_model_c_split")

print("Total users:", source.count())

source.groupBy("dataset_split").count().show()

In [0]:
# Reload validation users with their IDs
val_eval = (
    source
    .filter("dataset_split = 'VALIDATION'")
    .select(
        "third_party_id",
        "target",
        *feature_cols
    )
    .toPandas()
)

for col in numeric_cols:
    val_eval[col] = pd.to_numeric(
        val_eval[col], errors="coerce"
    )

# Generate predictions
val_eval["predicted_probability"] = (
    xgb_model.predict_proba(
        val_eval[feature_cols]
    )[:, 1]
)

# Fetch actual day-90 revenue for evaluation only
revenue_pdf = (
    spark.table("default.groww_modeling_dataset")
    .select(
        "third_party_id",
        "observable_revenue_90d"
    )
    .toPandas()
)

val_eval = val_eval.merge(
    revenue_pdf,
    on="third_party_id",
    how="left"
)

val_eval["observable_revenue_90d"] = (
    val_eval["observable_revenue_90d"].fillna(0)
)

# Select top 20 users
top20 = val_eval.nlargest(
    20,
    "predicted_probability"
)

total_revenue = val_eval["observable_revenue_90d"].sum()

revenue_captured = (
    top20["observable_revenue_90d"].sum() / total_revenue
    if total_revenue > 0 else 0
)

print("Top 20 users' actual day-90 revenue:",
      top20["observable_revenue_90d"].sum())

print("Total validation revenue:", total_revenue)

print("Revenue Captured@20:",
      round(revenue_captured * 100, 2), "%")

In [0]:
# Reload validation users with their IDs
val_eval = (
    source
    .filter("dataset_split = 'VALIDATION'")
    .select(
        "third_party_id",
        "target",
        *feature_cols
    )
    .toPandas()
)

for col in numeric_cols:
    val_eval[col] = pd.to_numeric(
        val_eval[col], errors="coerce"
    )

# Generate predictions
val_eval["predicted_probability"] = (
    xgb_model.predict_proba(
        val_eval[feature_cols]
    )[:, 1]
)

# Fetch actual day-90 revenue for evaluation only
revenue_pdf = (
    spark.table("default.groww_modeling_dataset")
    .select(
        "third_party_id",
        "observable_revenue_90d"
    )
    .toPandas()
)

val_eval = val_eval.merge(
    revenue_pdf,
    on="third_party_id",
    how="left"
)

val_eval["observable_revenue_90d"] = (
    val_eval["observable_revenue_90d"].fillna(0)
)

# Select top 20 users
top20 = val_eval.nlargest(
    20,
    "predicted_probability"
)

total_revenue = val_eval["observable_revenue_90d"].sum()

revenue_captured = (
    top20["observable_revenue_90d"].sum() / total_revenue
    if total_revenue > 0 else 0
)

print("Top 20 users' actual day-90 revenue:",
      top20["observable_revenue_90d"].sum())

print("Total validation revenue:", total_revenue)

print("Revenue Captured@20:",
      round(revenue_captured * 100, 2), "%")

In [0]:
display(
    top20[[
        "third_party_id",
        "predicted_probability",
        "target",
        "observable_revenue_90d"
    ]].sort_values(
        "predicted_probability",
        ascending=False
    )
)

In [0]:
xgb_results["Revenue_Captured_at_20"] = revenue_captured

display(pd.DataFrame([xgb_results]))

In [0]:
# Load test data
test = (
    source
    .filter("dataset_split = 'TEST'")
    .select(
        "third_party_id",
        "target",
        *feature_cols
    )
    .toPandas()
)

for col in numeric_cols:
    test[col] = pd.to_numeric(
        test[col], errors="coerce"
    )

X_test = test[feature_cols]
y_test = test["target"].astype(int)

# Generate predictions
test_prob = xgb_model.predict_proba(X_test)[:, 1]

test["predicted_probability"] = test_prob

# Top 20 users
top20_test = test.nlargest(20, "predicted_probability")

selected_high_value = int(top20_test["target"].sum())
total_high_value = int(y_test.sum())

precision = selected_high_value / 20
recall = selected_high_value / total_high_value
lift = precision / y_test.mean()

# Fetch day-90 revenue for evaluation only
revenue_pdf = (
    spark.table("default.groww_modeling_dataset")
    .select("third_party_id", "observable_revenue_90d")
    .toPandas()
)

test = test.merge(
    revenue_pdf,
    on="third_party_id",
    how="left"
)

test["observable_revenue_90d"] = (
    test["observable_revenue_90d"].fillna(0)
)

top20_test = test.nlargest(
    20, "predicted_probability"
)

total_revenue = test["observable_revenue_90d"].sum()

revenue_captured = (
    top20_test["observable_revenue_90d"].sum()
    / total_revenue
)

# Store test results
xgb_test_results = {
    "model": "XGBoost - Behaviour + Early Revenue",
    "ROC_AUC": roc_auc_score(y_test, test_prob),
    "PR_AUC": average_precision_score(y_test, test_prob),
    "Precision_at_20": precision,
    "Recall_at_20": recall,
    "Lift_at_20": lift,
    "Revenue_Captured_at_20": revenue_captured
}

display(pd.DataFrame([xgb_test_results]))

In [0]:
import pandas as pd

logistic_results = pd.DataFrame([
    {
        "model": "Model A - Behaviour Only",
        "ROC_AUC": 0.911503,
        "PR_AUC": 0.362591,
        "Precision_at_20": 0.65,
        "Recall_at_20": 0.25,
        "Lift_at_20": 26.55,
        "Revenue_Captured_at_20": 0.384599
    },
    {
        "model": "Model B - Behaviour + Revenue + Transaction Values",
        "ROC_AUC": 0.916877,
        "PR_AUC": 0.406711,
        "Precision_at_20": 0.70,
        "Recall_at_20": 0.269231,
        "Lift_at_20": 28.592308,
        "Revenue_Captured_at_20": 0.384003
    },
    {
        "model": "Model C - Behaviour + Early Revenue",
        "ROC_AUC": 0.916515,
        "PR_AUC": 0.427877,
        "Precision_at_20": 0.75,
        "Recall_at_20": 0.288462,
        "Lift_at_20": 30.634615,
        "Revenue_Captured_at_20": 0.397659
    }
])

display(logistic_results)

In [0]:
comparison = pd.concat(
    [
        logistic_results,
        pd.DataFrame([xgb_test_results])
    ],
    ignore_index=True
)

display(comparison)

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load corrected Model C dataset
source_c = spark.table("default.groww_model_c_split")

excluded = {
    "third_party_id", "signup_ts",
    "target", "dataset_split"
}

feature_cols_c = [
    c for c in source_c.columns if c not in excluded
]

numeric_types = (
    "int", "bigint", "double", "float",
    "decimal", "smallint", "tinyint"
)

numeric_cols_c = [
    f.name for f in source_c.schema.fields
    if f.name in feature_cols_c
    and f.dataType.simpleString().startswith(numeric_types)
]

categorical_cols_c = [
    c for c in feature_cols_c
    if c not in numeric_cols_c
]

pdf_c = source_c.select(
    "third_party_id", "signup_ts",
    "target", "dataset_split", *feature_cols_c
).toPandas()

for col in numeric_cols_c:
    pdf_c[col] = pd.to_numeric(
        pdf_c[col], errors="coerce"
    )

train_c = pdf_c[pdf_c.dataset_split == "TRAIN"].copy()
test_c = pdf_c[pdf_c.dataset_split == "TEST"].copy()

# Preprocessing
lr_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols_c),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(
            strategy="constant", fill_value="Unknown"
        )),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols_c)
])

# Train Logistic Regression
weekly_lr_model = Pipeline([
    ("preprocessor", lr_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

weekly_lr_model.fit(
    train_c[feature_cols_c],
    train_c["target"].astype(int)
)

print("Model C ready for weekly scoring.")

In [0]:
# Score test users
test_c["predicted_probability"] = (
    weekly_lr_model.predict_proba(
        test_c[feature_cols_c]
    )[:, 1]
)

# Day-7 eligibility date
test_c["day7_eligible_date"] = (
    pd.to_datetime(test_c["signup_ts"], utc=True)
    + pd.Timedelta(days=7)
)

# Week beginning Monday
test_c["week_start"] = (
    test_c["day7_eligible_date"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

# Rank users within each week
test_c["weekly_rank"] = (
    test_c.groupby("week_start")["predicted_probability"]
    .rank(method="first", ascending=False)
)

# Select top 20 each week
weekly_watchlist = test_c[
    test_c["weekly_rank"] <= 20
].copy()

print("Total eligible users:", len(test_c))
print("Total weekly watchlist selections:", len(weekly_watchlist))
print("Number of weeks:", weekly_watchlist["week_start"].nunique())

display(
    weekly_watchlist[[
        "third_party_id",
        "week_start",
        "predicted_probability",
        "weekly_rank",
        "target"
    ]].sort_values(["week_start", "weekly_rank"])
)

In [0]:
revenue_pdf = (
    spark.table("default.groww_modeling_dataset")
    .select(
        "third_party_id",
        "observable_revenue_90d"
    )
    .toPandas()
)

weekly_watchlist = weekly_watchlist.merge(
    revenue_pdf,
    on="third_party_id",
    how="left"
)

weekly_watchlist["observable_revenue_90d"] = (
    weekly_watchlist["observable_revenue_90d"].fillna(0)
)

# Overall weekly watchlist performance
selected_users = len(weekly_watchlist)
selected_high_value = int(weekly_watchlist["target"].sum())
total_high_value = int(test_c["target"].sum())

total_test_revenue = revenue_pdf[
    "observable_revenue_90d"
].sum()

selected_revenue = weekly_watchlist[
    "observable_revenue_90d"
].sum()

weekly_summary = {
    "Weeks": weekly_watchlist["week_start"].nunique(),
    "Selected_users": selected_users,
    "High_value_users_selected": selected_high_value,
    "Precision": selected_high_value / selected_users,
    "Recall": selected_high_value / total_high_value,
    "Revenue_Captured": selected_revenue / total_test_revenue
}

display(pd.DataFrame([weekly_summary]))